# Audio Similarity Search

### Environment Setup

In [6]:
# !python3 -m pip install datasets --user  # for data
# !python3 -m pip install librosa --user # play audio in python
# !python3 -m pip install panns-inference --user # audio embedding

### Load Libraries

In [66]:
import pinecone
from tqdm.autonotebook import tqdm
from datasets import load_dataset
from tqdm.auto import tqdm
import librosa # to load our own sound file
import panns_inference
import numpy as np

from IPython.display import Audio, display
import torch
from panns_inference import AudioTagging

### Load Data

Please see the data citation at the reference section. 

In [9]:
# broad 5 group of categories
# under each category total 10 classes
# under each class total 40 sample of sounds. 
# For Animal>>Dog >> 40 audio files, each around 5 seconds

data = load_dataset("ashraq/esc50", split="train")

Found cached dataset parquet (/home/mohsin/.cache/huggingface/datasets/ashraq___parquet/ashraq--esc50-1000c3b73cc1500f/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


In [10]:
data

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

In [16]:
data['audio'][0] # 50*40

{'path': None,
 'array': array([0., 0., 0., ..., 0., 0., 0.]),
 'sampling_rate': 44100}

In [18]:
data['audio'][0]['array']

array([0., 0., 0., ..., 0., 0., 0.])

In [17]:
sr = 44100 # 'sampling_rate': 44100

In [146]:
audios = np.array([file["array"] for file in data["audio"]]) # 2d array

### Audio EDA: Play Audios 

In [69]:
sound_num = 599 # 0 to 1999
sound_array = data[sound_num]["audio"]['array']
print(data[sound_num]["category"])
Audio(sound_array, rate=44100)

hand_saw


In [53]:
len(sound_array)  # -1 to 1

220500

In [54]:
audio_length = 5 # sec
sr = 44100 # sampling rate

samples = audio_length * sr

In [59]:
Audio(np.random.rand(220500), rate= sr) # white noise

### Initialize the Audio Embedding Model

In [67]:
# cpu or gpu
# import torch
# from panns_inference import AudioTagging

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model for Audio Tagging
model = AudioTagging(checkpoint_path=None, device = device)

Checkpoint path: /home/mohsin/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1


In [ ]:
_, sound_tagged = model.inference(sound_array[None, :])
# find dimension of vector
dim = sound_tagged.shape[1]

In [102]:
dim

2048

### Init Pinecone

In [141]:
# import pinecone

pinecone.init(api_key = API_KEY, environment=ENV)

# create index
index_name = "audio"
# pinecone.create_index(index_name,
#                      dimension=2048,
#                      metric='cosine') 

index = pinecone.Index(index_name)

In [142]:
pinecone.describe_index(index_name)

IndexDescription(name='audio', metric='cosine', replicas=1, dimension=2048.0, shards=1, pods=1, pod_type='p1.x1', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')

In [145]:
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

In [144]:
index.delete(delete_all=True)

{}

### Indexing

In [180]:
# from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0 , len(audios), batch_size)):
    
    i_end = min(len(audios), i+batch_size)
    batch = audios[i:i_end]
    # audio embedding/tagging
    _, emb = model.inference(batch)
    ids = [f"{id}" for id in range(i, i_end)] # string
    meta_data = [{"category": val} for val in data[i:i_end]["category"]]
    to_upsert = list(zip(ids, emb.tolist(), meta_data)) # tuple ("1", [], {"k":"v"})
    _ = index.upsert(vectors=to_upsert)


  0%|          | 0/32 [00:00<?, ?it/s]

In [150]:
meta_data = data[i:i_end]["category"]

In [161]:
# [{"category": val} for val in data[i:i_end]["category"]]

### Querying with in Sample Data: Data from example

In [171]:
audio_num = 559
print(data[audio_num]['category'])
Audio(data[audio_num]['audio']['array'], rate = 44100)


sea_waves


In [176]:
# embedding

qr_audio = data[audio_num]['audio']['array'] # column vector
qr_audio = qr_audio[None, :] # row vector


_, xq = model.inference(qr_audio)
xq.shape # row vector, np.array

(1, 2048)

In [183]:
# search

res = index.query(xq.tolist(), top_k=10, include_metadata=True)

In [191]:
res['matches'][0:3]

[{'id': '559',
  'metadata': {'category': 'sea_waves'},
  'score': 0.99999994,
  'values': []},
 {'id': '1091',
  'metadata': {'category': 'sea_waves'},
  'score': 0.970829,
  'values': []},
 {'id': '1424',
  'metadata': {'category': 'sea_waves'},
  'score': 0.967727661,
  'values': []}]

In [192]:
for item in res['matches']: 
    print(item['metadata']['category'], item['score'])

sea_waves 0.99999994
sea_waves 0.970829
sea_waves 0.967727661
wind 0.959257722
sea_waves 0.957942307
sea_waves 0.957576036
sea_waves 0.95682013
sea_waves 0.952222049
sea_waves 0.947814584
sea_waves 0.947782


In [208]:
for item in res["matches"][0:3]:
    idx = int(item["id"])
    print(item['metadata']['category'])
    display(Audio(data[idx]['audio']['array'], rate=44100))

sea_waves


sea_waves


sea_waves


### Querying with out of Sample Data: with our own data

In [216]:
# import librosa # to load our own sound file
# from IPython.display import Audio, display


# yamalyzer/audio/cough.wav
!wget https://storage.googleapis.com/audioset/yamalyzer/audio/cough.wav

test_audio, sr = librosa.load("cough.wav") # sound array and rate

--2023-06-11 13:34:09--  https://storage.googleapis.com/audioset/yamalyzer/audio/cough.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 2607:f8b0:400f:802::2010, 2607:f8b0:400f:804::2010, 2607:f8b0:400f:803::2010, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2607:f8b0:400f:802::2010|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 395088 (386K) [audio/wav]
Saving to: ‘cough.wav.3’

cough.wav.3         100%[===================>] 385.83K  --.-KB/s    in 0.1s    

2023-06-11 13:34:10 (2.63 MB/s) - ‘cough.wav.3’ saved [395088/395088]



In [217]:
Audio(test_audio, rate=sr)

In [220]:
# Query
_, xq = model.inference(test_audio[None,:])
res = index.query(xq.tolist(), 
                  top_k=3, 
                  include_metadata=True)

In [223]:
for item in res["matches"][0:3]:
    idx = int(item["id"])
    print(item['metadata']['category'], item['score'] )
    display(Audio(data[idx]['audio']['array'], rate=44100))

coughing 0.957787156


coughing 0.930912673


coughing 0.929008603


In [227]:
# !wget https://storage.googleapis.com/audioset/yamalyzer/audio/frogs.wav
test_audio, sr = librosa.load("frogs.wav") # sound array and rate

_, xq = model.inference(test_audio[None,:])
res = index.query(xq.tolist(), 
                  top_k=5, 
                  include_metadata=True)
for item in res["matches"][0:5]:
    idx = int(item["id"])
    print(item['metadata']['category'], item['score'] )
    display(Audio(data[idx]['audio']['array'], rate=44100))

crickets 0.86799413


crickets 0.831495821


frog 0.805249929


frog 0.79712081


crickets 0.794098139


In [229]:
Audio(test_audio, rate=sr)

### Reference

1. Audio Data Source: https://paperswithcode.com/dataset/esc-50 by Karol J. Piczak
2. ESC: Environmental Sound Classification
3. Data Source: https://github.com/karoldvl/ESC-50/archive/master.zip
4. Data Defination: https://github.com/karolpiczak/ESC-50 
5. Audio Tagging: https://github.com/qiuqiangkong/panns_inference 
6. Audio Data for testing: https://storage.googleapis.com/audioset 